In [ ]:
import numpy as np, pandas as pd, os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Experiment B3 — Phase 1 Projection Ablation

Exp 9/10 use Phase 1 projection trained on 8-9 analogy pairs, claiming it is 'cheap supervised projection.' But how many pairs are actually needed? And how sensitive is it to the lambda_compositionality weight? This ablation sweeps n_pairs (0-20) × lambda (0.25-4.0) and measures MRR + Probe 5 accuracy.

In [ ]:
!pip install sentence-transformers torch matplotlib pandas -q

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
torch.manual_seed(42)

In [ ]:
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('LaBSE', device='cpu')

CONCEPTS = [
    'water','fire','earth','sky','wind','stone','river','mountain','forest','child',
    'elder','friend','enemy','time','life','death','peace','war','hope','dream',
    'change','love','fear','trust','joy','pain','give','take','speak','think',
    'find','lose','build','break','eat','feel','sleep','run','light','dark',
    'teach','learn','kill','grow','fall','rise','sing','cry',
    'knife','hand','voice','home','silence','memory','freedom','beauty',
]
SPEAKER_WEIGHTS = {'en':1500,'zh':1100,'es':560,'ar':380,'ru':260}
LANG_VOCAB = {
    'en': CONCEPTS,
    'zh': ['水','火','土','天空','风','石头','河流','山','森林','孩子','老人','朋友','敌人','时间','生命','死亡','和平','战争','希望','梦想','改变','爱','恐惧','信任','喜悦','痛苦','给','拿','说话','思考','找到','失去','建造','破坏','吃','感觉','睡觉','跑','光','暗','教','学习','杀','生长','落下','升起','唱歌','哭','刀','手','声音','家','沉默','记忆','自由','美丽'],
    'es': ['agua','fuego','tierra','cielo','viento','piedra','río','montaña','bosque','niño','anciano','amigo','enemigo','tiempo','vida','muerte','paz','guerra','esperanza','sueño','cambio','amor','miedo','confianza','alegría','dolor','dar','tomar','hablar','pensar','encontrar','perder','construir','romper','comer','sentir','dormir','correr','luz','oscuridad','enseñar','aprender','matar','crecer','caer','subir','cantar','llorar','cuchillo','mano','voz','hogar','silencio','memoria','libertad','belleza'],
    'ar': ['ماء','نار','أرض','سماء','ريح','حجر','نهر','جبل','غابة','طفل','مسن','صديق','عدو','وقت','حياة','موت','سلام','حرب','أمل','حلم','تغير','حب','خوف','ثقة','فرح','ألم','أعطى','أخذ','تكلم','فكر','وجد','فقد','بنى','كسر','أكل','شعر','نوم','ركض','ضوء','ظلام','علم','تعلم','قتل','نما','سقط','صعد','غنى','بكى','سكين','يد','صوت','بيت','صمت','ذاكرة','حرية','جمال'],
    'ru': ['вода','огонь','земля','небо','ветер','камень','река','гора','лес','ребёнок','старик','друг','враг','время','жизнь','смерть','мир','война','надежда','мечта','изменение','любовь','страх','доверие','радость','боль','давать','брать','говорить','думать','найти','потерять','строить','ломать','есть','чувствовать','спать','бежать','свет','тьма','учить','учиться','убить','расти','падать','подниматься','петь','плакать','нож','рука','голос','дом','тишина','память','свобода','красота'],
}

print('Embedding vocabulary...')
embs = {}
for lang, words in LANG_VOCAB.items():
    embs[lang] = model.encode(words, normalize_embeddings=True, show_progress_bar=False)

total_w = sum(SPEAKER_WEIGHTS.values())
centroid_matrix = sum(SPEAKER_WEIGHTS[l]/total_w * embs[l] for l in SPEAKER_WEIGHTS)
centroid_matrix /= np.linalg.norm(centroid_matrix, axis=1, keepdims=True)+1e-9
print('Centroid matrix:', centroid_matrix.shape)

ALL_ANALOGY_PAIRS = [
    ('love','fear','joy','pain'),('life','death','peace','war'),
    ('give','take','build','break'),('light','dark','hope','pain'),
    ('friend','enemy','peace','war'),('find','lose','give','take'),
    ('rise','fall','build','break'),('sing','cry','joy','pain'),
    ('teach','learn','give','take'),('hope','fear','life','death'),
    ('love','fear','friend','enemy'),('water','fire','light','dark'),
    ('speak','think','give','take'),('build','break','grow','kill'),
    ('memory','dream','find','lose'),('freedom','silence','peace','war'),
    ('beauty','pain','joy','fear'),('home','forest','peace','war'),
    ('hand','knife','give','take'),('voice','silence','speak','think'),
]
VALID_PAIRS = [(a,b,c,d) for a,b,c,d in ALL_ANALOGY_PAIRS
               if all(w in CONCEPTS for w in [a,b,c,d])]
print('Valid analogy pairs:', len(VALID_PAIRS))

In [ ]:
class Proj(nn.Module):
    def __init__(self):
        super().__init__()
        self.W = nn.Linear(768, 768, bias=False)
        nn.init.eye_(self.W.weight)
    def forward(self, x): return F.normalize(self.W(x), dim=-1)

def compute_mrr(matrix, pairs):
    rranks = []
    for a,b,c,d in pairs:
        if not all(w in CONCEPTS for w in [a,b,c,d]): continue
        ia,ib,ic,id_ = [CONCEPTS.index(w) for w in [a,b,c,d]]
        pred = matrix[ia]-matrix[ib]+matrix[ic]
        pred = pred/np.linalg.norm(pred).clip(1e-9)
        sims = matrix@pred
        for x in (ia,ib,ic): sims[x]=-np.inf
        rank = 1+int((sims>sims[id_]).sum())
        rranks.append(1.0/rank)
    return float(np.mean(rranks)) if rranks else 0.0

def probe5_acc(matrix):
    # Test if opposition direction generalises: use held-out pairs not in training
    TEST_PAIRS = [('light','dark','give','take'),('joy','pain','love','fear'),
                  ('rise','fall','hope','fear'),('friend','enemy','build','break')]
    TEST_PAIRS = [(a,b,c,d) for a,b,c,d in TEST_PAIRS if all(w in CONCEPTS for w in [a,b,c,d])]
    if not TEST_PAIRS: return 0.0
    correct = 0
    for a,b,c,d in TEST_PAIRS:
        ia,ib,ic,id_ = [CONCEPTS.index(w) for w in [a,b,c,d]]
        opp = matrix[ib]-matrix[ia]; opp /= np.linalg.norm(opp)+1e-9
        target = matrix[ic]+opp; target /= np.linalg.norm(target)+1e-9
        sims = matrix@target
        for x in (ia,ib,ic): sims[x]=-np.inf
        if sims.argmax()==id_: correct+=1
    return correct/len(TEST_PAIRS)

def train_projection(n_pairs, lam_comp, n_epochs=800):
    if n_pairs == 0: return centroid_matrix.copy()
    pairs = VALID_PAIRS[:n_pairs]
    proj = Proj(); opt = torch.optim.Adam(proj.parameters(), lr=5e-4)
    X = torch.tensor(centroid_matrix, dtype=torch.float32)
    for epoch in range(n_epochs):
        P = proj(X)
        L_prox = 1.0 - (P*X).sum(-1).mean()
        L_comp = torch.tensor(0.0); n=0
        for a,b,c,d in pairs:
            ia,ib,ic,id_ = [CONCEPTS.index(w) for w in [a,b,c,d]]
            pred = F.normalize(P[ia]-P[ib]+P[ic], dim=0)
            L_comp = L_comp + (1.0-(pred*P[id_]).sum()); n+=1
        L_comp = L_comp/n
        loss = L_prox + lam_comp*L_comp
        opt.zero_grad(); loss.backward(); opt.step()
    proj.eval()
    with torch.no_grad():
        return proj(X).numpy()

raw_mrr = compute_mrr(centroid_matrix, VALID_PAIRS)
raw_p5  = probe5_acc(centroid_matrix)
print('Raw LaBSE: MRR={:.3f}  Probe5={:.3f}'.format(raw_mrr, raw_p5))

N_PAIRS_SWEEP = [0, 2, 3, 5, 9, 15, min(20, len(VALID_PAIRS))]
LAM_SWEEP     = [0.25, 0.5, 1.0, 2.0, 4.0]

results = []
for n_pairs in N_PAIRS_SWEEP:
    for lam in LAM_SWEEP:
        projected = train_projection(n_pairs, lam)
        mrr = compute_mrr(projected, VALID_PAIRS)
        p5  = probe5_acc(projected)
        results.append({'n_pairs':n_pairs,'lam':lam,'mrr':round(mrr,4),'probe5':round(p5,4)})
        print('  n_pairs={:2d}  lam={:.2f}: MRR={:.3f}  Probe5={:.3f}'.format(n_pairs, lam, mrr, p5))

import pandas as pd
df = pd.DataFrame(results)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
COLORS = ['#E24B4A','#378ADD','#1D9E75','#EF9F27','#7F77DD']

for ax, metric, title in [
    (axes[0], 'mrr',    'Analogy MRR'),
    (axes[1], 'probe5', 'Probe 5 Accuracy (opposition)'),
]:
    pivot = df.pivot(index='n_pairs', columns='lam', values=metric)
    baseline = raw_mrr if metric=='mrr' else raw_p5
    for lam, color in zip(LAM_SWEEP, COLORS):
        if lam in pivot.columns:
            ax.plot(pivot.index, pivot[lam], 'o-', color=color, linewidth=2, label='lambda={:.2f}'.format(lam))
    ax.axhline(baseline, color='gray', linestyle='--', label='Raw LaBSE ({:.3f})'.format(baseline))
    ax.set_xlabel('Number of analogy pairs'); ax.set_ylabel(metric.upper())
    ax.set_title('{} vs n_pairs\n(How few pairs needed?)'.format(title))
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Experiment B3 — Phase 1 Projection Ablation\n'
             'Quantifying how cheap "cheap supervised projection" really is',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp_b3_phase1_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*60)
print('EXPERIMENT B3 — SUMMARY')
print('='*60)
print('Raw baseline: MRR={:.3f}  Probe5={:.3f}'.format(raw_mrr, raw_p5))
best = df.nlargest(1,'mrr').iloc[0]
print('Best MRR: {:.3f} at n_pairs={:d} lam={:.2f}'.format(best.mrr, int(best.n_pairs), best.lam))
min_pairs_df = df[df.mrr > raw_mrr + 0.5*(df.mrr.max()-raw_mrr)]
if not min_pairs_df.empty:
    min_pairs = int(min_pairs_df.nsmallest(1,'n_pairs').iloc[0].n_pairs)
    print('Minimum pairs for >50% of max gain: {:d}'.format(min_pairs))
print()
print('MRR heatmap:')
print(df.pivot(index='n_pairs',columns='lam',values='mrr').round(3).to_string())